In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q20/q20_rewrite/checkpoints/pre_cell_5.pickle

In [4]:
%%RecordEvent
%%time
### cell 5 ###
# Filter predicates using GPU‐friendly operations (no pd.Timestamp scalars)
psel = part.P_NAME.str.startswith("azure")
nsel = nation.N_NAME == "JORDAN"
lsel = (
    lineitem.L_SHIPDATE >= "1996-01-01"
) & (
    lineitem.L_SHIPDATE < "1997-01-01"
)

fpart = part[psel]
fnation = nation[nsel]
flineitem = lineitem[lsel]

jn1 = fpart.merge(partsupp, left_on="P_PARTKEY", right_on="PS_PARTKEY")
jn2 = jn1.merge(
    flineitem,
    left_on=["PS_PARTKEY", "PS_SUPPKEY"],
    right_on=["L_PARTKEY", "L_SUPPKEY"],
)

gb = jn2.groupby(
    ["PS_PARTKEY", "PS_SUPPKEY", "PS_AVAILQTY"],
    as_index=False,
    sort=False
)["L_QUANTITY"].sum()

# retain only those with available quantity > half of shipped quantity
gbsel = gb.PS_AVAILQTY > (0.5 * gb.L_QUANTITY)
fgb = gb[gbsel]

jn3 = fgb.merge(supplier, left_on="PS_SUPPKEY", right_on="S_SUPPKEY")
jn4 = fnation.merge(jn3, left_on="N_NATIONKEY", right_on="S_NATIONKEY")[["S_NAME", "S_ADDRESS"]]

total = jn4.sort_values("S_NAME").drop_duplicates()

CPU times: user 102 ms, sys: 19.1 ms, total: 121 ms
Wall time: 144 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q20/rewritten/q20_small_rewrite/checkpoints/post_cell_5_try_0.pickle